# 8. Chains and partial relations

Reads through several relations in a row, and through relations with missing neighbors, continuing "2. Connectivities": neighbor ancestry, siblings, occurrences, partial relations with boundary values, and the read chains through which the backends propagate domains.

The example mesh: a triangle split in four, the interior cell `c0` and the boundary cells `c1`, `c2`, `c3`, with 9 edges and 6 vertices. The 6 boundary edges have one cell: the other one is missing.

```
              v2
             /  \
           e7    e6
           / c3   \
         v5 --e1-- v4
         / \  c0  / \
       e8  e2   e0   e5
       / c1  \  /  c2 \
     v0 --e3-- v3 --e4-- v1
```

Contents: the mesh and partial relations · chains and neighbor ancestry · siblings and occurrences · partial relations and boundary values · read chains and extents · summary.

## The mesh and partial relations

| Concept | Kind | Meaning |
|---|---|---|
| `partial=True` | `Relation` keyword | Neighbors can be missing at the boundary of the mesh, e.g. the second cell of a boundary edge |
| `MISSING_VALUE` | constant | `-2`: a missing neighbor in a table (only if `R.partial`), unlike `SKIP_VALUE`, `-1`, a shorter row (only if `R.skip_values`) |
| `Mesh(tables, extents=...)` | class | The tables, each checked once, with the number of points of some dimensions: the row counts and the neighbor indices are checked against them |

`E2C` is partial: each boundary edge has one cell, then `MISSING_VALUE`. The mesh is built once, then provided by `with split:` (see "2. Connectivities").

In [1]:
import numpy as np
from concepts import MISSING_VALUE, Dims, Field, Location, Mesh, Neighbor, Relation
from implementation import NumPyArrayField, neighbor_table, sum_over


class Cell(Location): ...


class Edge(Location): ...


class Vertex(Location): ...


class C2E(Relation[Cell, Edge], max_neighbors=3): ...


class C2V(Relation[Cell, Vertex], max_neighbors=3): ...


class E2V(Relation[Edge, Vertex], max_neighbors=2): ...


class E2C(Relation[Edge, Cell], max_neighbors=2, partial=True): ...  # boundary edges: 1 cell


M = MISSING_VALUE
tables = {
    C2E: neighbor_table(C2E, [[0, 1, 2], [3, 2, 8], [4, 5, 0], [1, 6, 7]]),
    C2V: neighbor_table(C2V, [[3, 4, 5], [0, 3, 5], [3, 1, 4], [5, 4, 2]]),
    E2V: neighbor_table(
        E2V, [[3, 4], [4, 5], [5, 3], [0, 3], [3, 1], [1, 4], [4, 2], [2, 5], [5, 0]]
    ),
    E2C: neighbor_table(
        E2C, [[0, 2], [0, 3], [0, 1], [1, M], [2, M], [2, M], [3, M], [3, M], [1, M]]
    ),
}
extents = {Cell: 4, Edge: 9, Vertex: 6}
split = Mesh(tables, extents=extents)
print("E2C =", split[E2C])

t = NumPyArrayField({Cell: range(4)}, [20.0, 10.0, 30.0, 40.0])  # cell temperatures
outside = 15.0  # the temperature beyond the boundary
vid = NumPyArrayField({Vertex: range(6)}, np.arange(6.0))  # vertex ids
e = NumPyArrayField({Edge: range(9)}, np.arange(1.0, 10.0))  # 1, ..., 9

for bad in (
    {**tables, C2V: neighbor_table(C2V, [[3, 4, 5], [0, 3, 5], [3, 1, 4], [5, 4, 6]])},
    {**tables, C2E: neighbor_table(C2E, [[0, 1, M], [3, 2, 8], [4, 5, 0], [1, 6, 7]])},
):  # no vertex 6; C2E is not partial
    try:
        Mesh(bad, extents=extents)
    except ValueError as error:
        print("ValueError:", error)

E2C = Field(Edge[0:9], Neighbor[E2C][0:2]: [[0, 2], [0, 3], [0, 1], [1, -2], [2, -2], [2, -2], [3, -2], [3, -2], [1, -2]])
ValueError: The table of 'C2V' has neighbor indices beyond the extent of 'Vertex', 6.
ValueError: The table of 'C2E' has invalid neighbor indices.


## Chains and neighbor ancestry

A gather through a relation `R: O -> T`, on a field with neighbor dimensions hanging from `T`, first renames them, recursively: `Neighbor[X, T]` becomes `Neighbor[X, Neighbor[R, O]]` (*reparenting*: the data is the same). Then `T` is replaced by `O`, followed by `Neighbor[R, O]`. So `vid(E2V)(C2E)` has the dimensions `(Cell, Neighbor[C2E], Neighbor[E2V, Neighbor[C2E]])`: the vertices of each edge of each cell.

| Concept | Kind | Meaning |
|---|---|---|
| `Neighbor[R, P]` | dimension | The neighbor dimension of `R` hanging from its parent `P`: `R.source`, a root (the default: `Neighbor[C2E]` is `Neighbor[C2E, Cell]`), or a neighbor dimension of a relation to `R.source` (else a `TypeError`) |
| `.parent`, `.relation` | attributes | The parent and the relation of a neighbor dimension |
| `root_of(dim)` | function | The root of a neighbor dimension, following its parents: `Cell` for `Neighbor[E2V, Neighbor[C2E]]` |
| `neighbor_of(R, P)` | function | `Neighbor[R, P]`, for a relation known at runtime only |

**The leaf rule:** `sum_over(f, axis=S)` needs a leaf `S`: no other neighbor dimension of `f` hangs from it (else a `ValueError`). So the vertices of each edge are summed first, then the edges: each vertex of a cell is counted once per edge of the cell it is on, twice in a triangle.

In [2]:
from concepts import neighbor_of, root_of

with split:
    chain = vid(E2V)(C2E)  # the vertices of each edge of each cell
    print("dims:", [d.__name__ for d in chain.domain])
    print("cell 1:", chain.data[1].tolist())  # e3, e2, e8: [v0, v3], [v5, v3], [v5, v0]
    S = Neighbor[E2V, Neighbor[C2E]]
    print(
        S.__name__,
        S.parent.__name__,
        S.relation.__name__,
        root_of(S).__name__,
        S is neighbor_of(E2V, Neighbor[C2E]),
    )
    try:
        sum_over(chain, axis=Neighbor[C2E])  # not a leaf
    except ValueError as error:
        print("ValueError:", error)
    two_hops = sum_over(sum_over(chain, axis=S), axis=Neighbor[C2E])  # the leaf first
    # twice the vertex ids: cell 0, v3 + v4 + v5 = 12 -> 24; cell 1, v0 + v3 + v5 = 8 -> 16
    print("two hops =", two_hops)
    print("C2V      =", sum_over(vid(C2V), axis=Neighbor[C2V]))
try:
    Neighbor[E2V, Neighbor[C2V]]  # C2V ends at vertices, but E2V starts at edges
except TypeError as error:
    print("TypeError:", error)

dims: ['Cell', 'Neighbor[C2E]', 'Neighbor[E2V, Neighbor[C2E]]']
cell 1: [[0.0, 3.0], [5.0, 3.0], [5.0, 0.0]]
Neighbor[E2V, Neighbor[C2E]] Neighbor[C2E] E2V Cell True
ValueError: Reduce over ['Neighbor[E2V, Neighbor[C2E]]'] before 'Neighbor[C2E]', which they hang from.
two hops = Field(Cell[0:4]: [24.0, 16.0, 16.0, 22.0])
C2V      = Field(Cell[0:4]: [12.0, 8.0, 8.0, 11.0])
TypeError: 'Neighbor[C2V]' cannot be the parent of a neighbor of 'E2V': it must be 'Edge' or a neighbor of a relation to it.


## Siblings and occurrences

Neighbor dimensions of different relations from the same root are *siblings*: arithmetic broadcasts them (see "1. Fields and domains"), so `e(C2E) * vid(C2V)` has the dimensions `(Cell, Neighbor[C2E], Neighbor[C2V])`: every edge with every vertex of each cell. A field operator returns its result in the dimension order of its annotation, on every backend.

Occurrences are nominal: a subclass of a relation, `class C2E_b(C2E): ...`, is another *occurrence* of it, with the same table but its own neighbor dimension. So `e(C2E) * e(C2E_b)` pairs every two edges of a cell (an outer product per cell), while `e(C2E) * e(C2E)` pairs each edge with itself.

In [3]:
import contextlib
import io

from implementation import column_major, compiled, eager, field_operator, lazy, lazy_backward

with split:
    siblings = e(C2E) * vid(C2V)
    print("siblings:", [d.__name__ for d in siblings.domain], siblings.data.shape)


@field_operator()
def edge_vertex(
    e: Field[Dims[Edge], float], v: Field[Dims[Vertex], float]
) -> Field[Dims[Cell, Neighbor[C2V], Neighbor[C2E]], float]:
    return e(C2E) * v(C2V)  # (Cell, Neighbor[C2E], Neighbor[C2V]), returned as annotated


with split:
    for backend in (eager, lazy, lazy_backward, column_major, compiled):
        with contextlib.redirect_stdout(io.StringIO()):  # silence the backends
            result = backend(edge_vertex)(e, vid)
            dims, values = [d.__name__ for d in result.domain], result.data[0, 0].tolist()
        print(f"{backend}: {dims} {values}")  # cell 0, its vertex v3, by e0, e1, e2: 3 * [1, 2, 3]


class C2E_b(C2E): ...  # another occurrence of C2E


with split:
    outer = e(C2E) * e(C2E_b)  # (Cell, Neighbor[C2E], Neighbor[C2E_b])
    # the square of the sum: cell 0, (1 + 2 + 3) ** 2 = 36; cell 1, (4 + 3 + 9) ** 2 = 256
    print("outer   =", sum_over(sum_over(outer, axis=Neighbor[C2E_b]), axis=Neighbor[C2E]))
    # the sum of the squares: cell 0, 1 + 4 + 9 = 14; cell 1, 16 + 9 + 81 = 106
    print("aligned =", sum_over(e(C2E) * e(C2E), axis=Neighbor[C2E]))

siblings: ['Cell', 'Neighbor[C2E]', 'Neighbor[C2V]'] (4, 3, 3)
eager: ['Cell', 'Neighbor[C2V]', 'Neighbor[C2E]'] [3.0, 6.0, 9.0]
lazy: ['Cell', 'Neighbor[C2V]', 'Neighbor[C2E]'] [3.0, 6.0, 9.0]
lazy_backward: ['Cell', 'Neighbor[C2V]', 'Neighbor[C2E]'] [3.0, 6.0, 9.0]
column_major: ['Cell', 'Neighbor[C2V]', 'Neighbor[C2E]'] [3.0, 6.0, 9.0]
compiled: ['Cell', 'Neighbor[C2V]', 'Neighbor[C2E]'] [3.0, 6.0, 9.0]
outer   = Field(Cell[0:4]: [36.0, 256.0, 144.0, 289.0])
aligned = Field(Cell[0:4]: [14.0, 106.0, 62.0, 117.0])


## Partial relations and boundary values

Reading through a partial relation needs either a domain without the missing neighbors, or a value for them:

- `t(E2C)` is defined on the edges whose cells are all present and in the domain of `t`: here the interior edges, `Edge[0:3]`;
- `t(E2C.boundary(15.0))` reads `15.0` at the missing neighbors: it is defined on the edges whose other cells are in the domain of `t`, with the same `Neighbor[E2C]`. The value may be an argument of a DSL operator. A relation that is not partial has no `.boundary` (a `TypeError`).

Reductions mask only the skip positions, never the missing ones: a boundary value counts. Backward, a plain read of missing neighbors raises a `ValueError`: the request is beyond the computable domain. `neighbors(R, sources, boundary=True)` and `sources_within(R, targets, boundary=True)` propagate domains for boundary reads (see "2. Connectivities").

Phase 1 limit: boundary reads, like functional ones (see "9. Footprint algebra and pairs"), only read fields without neighbor dimensions (else a `NotImplementedError`).

In [4]:
from implementation import neighbors, sources_within

with split:
    print("plain    =", t(E2C))  # the interior edges e0-e2 only
    print("boundary =", t(E2C.boundary(outside)))
    # the mean of the two sides of each edge: e0, (20 + 30) / 2 = 25; e3, (10 + 15) / 2 = 12.5
    print("mean     =", 0.5 * sum_over(t(E2C.boundary(outside)), axis=Neighbor[E2C]))
    try:
        neighbors(E2C, (0, 3))  # e3 has a missing cell
    except ValueError as error:
        print("ValueError:", error)
    print("cells of e0, e3, with a boundary value:", neighbors(E2C, (0, 3), boundary=True))
    print(
        "edges within all the cells:",
        sources_within(E2C, range(4)),
        "| with a boundary value:",
        sources_within(E2C, range(4), boundary=True),
    )
    try:
        C2E.boundary(0.0)
    except TypeError as error:
        print("TypeError:", error)
    loc = t(E2C)(C2E)  # (Cell, Neighbor[C2E], Neighbor[E2C, Neighbor[C2E]]), on c0
    try:
        loc(E2C.boundary(1.0))
    except NotImplementedError as error:
        print("NotImplementedError:", error)

plain    = Field(Edge[0:3], Neighbor[E2C][0:2]: [[20.0, 30.0], [20.0, 40.0], [20.0, 10.0]])
boundary = Field(Edge[0:9], Neighbor[E2C][0:2]: [[20.0, 30.0], [20.0, 40.0], [20.0, 10.0], [10.0, 15.0], [30.0, 15.0], [30.0, 15.0], [40.0, 15.0], [40.0, 15.0], [10.0, 15.0]])
mean     = Field(Edge[0:9]: [25.0, 30.0, 15.0, 12.5, 22.5, 22.5, 27.5, 27.5, 12.5])
ValueError: Edge[3:4] is beyond the computable domain: it has missing neighbors through 'E2C' (read with `E2C.boundary(value)` there).
cells of e0, e3, with a boundary value: range(0, 3)
edges within all the cells: range(0, 3) | with a boundary value: range(0, 9)
TypeError: 'C2E' is not partial: it has no boundary.
NotImplementedError: `E2C.boundary(...)` reads fields without neighbor dimensions only, not Cell[0:1], Neighbor[C2E][0:3], Neighbor[E2C, Neighbor[C2E]][0:2].


## Read chains and extents

The `Extents` of a DSL operator are chains of reads (see "3. DSL definitions and operators"): a read through relations in a row is a chain of steps, with `Boundary(E2C)` for a boundary read, without its value (an argument, only known when called); reads are followed through local variables. The lazy backends propagate domains through the chains, step by step: forward, `forward_domain` gives the points computable from the domain of an argument; backward, `backward_domain` gives the points of an argument needed for a requested domain, and raises a `ValueError` for a plain read of missing neighbors.

The example: the heat across the edges of each cell, the sum over its edges of the temperatures of both their cells, with the outside temperature beyond the boundary.

In [5]:
from concepts import Boundary, Reduction
from implementation import backward_domain, format_domain, forward_domain


def format_step(step):
    if isinstance(step, type):  # a relation
        return step.__name__
    if isinstance(step, Boundary):
        return f"Boundary({step.relation.__name__})"
    if isinstance(step, Reduction):
        return f"Reduction({step.axis.__name__})"
    return f"{step[0].__name__}{step[1]:+d}"


def format_reads(reads):
    return ["(" + ", ".join(map(format_step, read)) + ")" for read in reads]


@field_operator()
def across_edges(t: Field[Dims[Cell], float], outside: float) -> Field[Dims[Cell], float]:
    sides = t(E2C.boundary(outside))(C2E)  # (Cell, Neighbor[C2E], Neighbor[E2C, Neighbor[C2E]])
    per_edge = sum_over(sides, axis=Neighbor[E2C, Neighbor[C2E]])  # the leaf first
    return sum_over(per_edge, axis=Neighbor[C2E])


@field_operator()
def across_plain(t: Field[Dims[Cell], float]) -> Field[Dims[Cell], float]:
    return sum_over(sum_over(t(E2C)(C2E), axis=Neighbor[E2C, Neighbor[C2E]]), axis=Neighbor[C2E])


reads = across_edges.__gt_extents__["t"]
print("across_edges reads t with", *format_reads(reads))
with split:
    # c1 and c3: their edges e3, e2, e8 and e1, e6, e7, whose cells are c0, c1 and c3
    print("t needed for Cell{1, 3}:", format_domain(backward_domain({Cell: (1, 3)}, reads)))
    computable = forward_domain({Cell: (0, 1, 3)}, reads)  # not c0: its edge e0 needs c2
    print("computable from Cell{0, 1, 3}:", format_domain(computable))
    try:
        backward_domain({Cell: (1,)}, across_plain.__gt_extents__["t"])  # e3, e8: missing
    except ValueError as error:
        print("ValueError:", error)
    # c0: e0, 20 + 30, e1, 20 + 40, e2, 20 + 10 -> 140; c1: e3, 10 + 15, e2, 30, e8, 25 -> 80
    print("across_edges =", across_edges(t, outside))
    print("across_plain =", across_plain(t))  # only c0, whose edges are all interior

across_edges reads t with (Boundary(E2C), C2E, Reduction(Neighbor[E2C, Neighbor[C2E]]), Reduction(Neighbor[C2E]))
t needed for Cell{1, 3}: Cell{0, 1, 3}
computable from Cell{0, 1, 3}: Cell{1, 3}
ValueError: Edge{3, 8} is beyond the computable domain: it has missing neighbors through 'E2C' (read with `E2C.boundary(value)` there).
across_edges = Field(Cell[0:4]: [140.0, 80.0, 140.0, 170.0])
across_plain = Field(Cell[0:1]: [140.0])


In [6]:
part = t[{Cell: (0, 1, 3)}]  # no temperature for c2
results = {}
with split:
    for backend in (eager, lazy, lazy_backward, column_major, compiled):
        results[backend] = backend(across_edges)[{Cell: (1, 3)}](part, outside)
for backend, result in results.items():  # the lazy backends compute their values now
    print(f"--- {backend}: {result}")
with split:
    try:
        eager(across_edges)[{Cell: (2,)}](part, outside)  # c2 is not computable
    except ValueError as error:
        print("ValueError:", error)

[eager] computed Field(Cell{1, 3}: [80.0, 170.0])
[allocated] computed Field(Cell{1, 3}: [80.0, 170.0])
[compiled] compiling across_edges
[compiled] computed Field(Cell{1, 3}: [80.0, 170.0])
--- eager: Field(Cell{1, 3}: [80.0, 170.0])
[lazy] computed Cell{1, 3}, kept Cell{1, 3}
--- lazy: Field(Cell{1, 3}: [80.0, 170.0])
[lazy_backward] computed Cell{1, 3}
--- lazy_backward: Field(Cell{1, 3}: [80.0, 170.0])
--- column_major: Field(Cell{1, 3}: [80.0, 170.0])
--- compiled: Field(Cell{1, 3}: [80.0, 170.0])
[eager] computed Field(Cell{1, 3}: [80.0, 170.0])
ValueError: Cell{2} is not within the domain Cell{1, 3}.


## Summary

| Feature | What is checked |
|---|---|
| `Neighbor[R, P]`, chains `vid(E2V)(C2E)` | the parent of a neighbor dimension (a `TypeError`); the leaf rule of reductions (a `ValueError`) |
| Siblings, `e(C2E) * vid(C2V)` | broadcast; results in the dimension order of the annotation, on every backend |
| Occurrences, `class C2E_b(C2E)` | no configuration of their own (a `TypeError`); the table of their relation, their own neighbor dimension |
| `partial=True`, `MISSING_VALUE` | only in the tables of partial relations; rows and indices against the `extents` of the mesh (a `ValueError`) |
| `R.boundary(value)` | only for partial relations (a `TypeError`); only on fields without neighbor dimensions (a `NotImplementedError`) |
| Read chains, `forward_domain`, `backward_domain` | backward, plain reads of missing neighbors (a `ValueError`); requested domains within the result's (a `ValueError`) |